<a href="https://colab.research.google.com/github/gez2code/dermamnist-hybrid-study/blob/main/Experiment_Setup_Modular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# ==========================================
# 1. INSTALL & IMPORT DEPENDENCIES
# ==========================================
# We install 'medmnist' for the dataset and 'wandb' for experiment tracking.
!pip install medmnist wandb -q

import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from medmnist import DermaMNIST
import wandb
from wandb.integration.keras import WandbMetricsLogger
import matplotlib.pyplot as plt

# Metrics and utilities for evaluation
from sklearn.utils import class_weight
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# ==========================================
# 2. REPRODUCIBILITY SETUP
# ==========================================
# Setting seeds ensures that if we run the same code twice, we get the same results.
# This is crucial for scientific comparisons between models.
SEED = 42
def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)

set_seeds()
print(f"Random seed set to {SEED}")

# ==========================================
# 3. DATA LOADING & PROCESSING
# ==========================================
def load_and_prep_data():
    print("Loading DermaMNIST...")
    # 'size=28' ensures we respect the native resolution (preventing false upscaling)
    train_data = DermaMNIST(split='train', download=True, size=28)
    val_data = DermaMNIST(split='val', download=True, size=28)
    test_data = DermaMNIST(split='test', download=True, size=28)

    # Normalization: Scale pixel values to 0-1 range for faster convergence
    x_train, y_train = train_data.imgs.astype('float32')/255.0, train_data.labels
    x_val, y_val = val_data.imgs.astype('float32')/255.0, val_data.labels
    x_test, y_test = test_data.imgs.astype('float32')/255.0, test_data.labels

    # Binary Mapping: Grouping specific classes into 'Illness' (1) vs 'Benign' (0)
    # This simplifies the problem from 7 classes to a binary screening task.
    to_binary = lambda y: np.isin(y, [0, 1, 6]).astype(int)
    y_train_bin, y_val_bin, y_test_bin = to_binary(y_train), to_binary(y_val), to_binary(y_test)

    # One-Hot Encoding: Converts [0, 1] to [[1,0], [0,1]] for categorical_crossentropy
    y_train_enc = tf.keras.utils.to_categorical(y_train_bin, 2)
    y_val_enc = tf.keras.utils.to_categorical(y_val_bin, 2)
    y_test_enc = tf.keras.utils.to_categorical(y_test_bin, 2)

    # Handling Imbalance: Compute weights so the model pays more attention to rare classes.
    # This is critical for medical datasets where 'Illness' is often the minority.
    cw = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_bin),
        y=y_train_bin.flatten()
    )
    weights = {0: cw[0], 1: cw[1]}

    print(f"Data Loaded. Class Weights: {weights}")
    return (x_train, y_train_enc), (x_val, y_val_enc), (x_test, y_test_enc), weights

# Load Data Once to be used by all experiments
(x_train, y_train), (x_val, y_val), (x_test, y_test), class_weights = load_and_prep_data()

# ==========================================
# 4. UNIVERSAL TRAINER FUNCTION
# ==========================================
# This function handles the entire lifecycle of an experiment:
# Init W&B -> Build Model -> Train -> Evaluate -> Log Results
def train_experiment(model_builder, exp_name, config):
    # --- A. Setup Weights & Biases ---
    if wandb.run is not None: wandb.finish() # Clean up any previous runs
    wandb.init(project="DermaMNIST_Project", name=exp_name, config=config)

    # --- B. Build Model with Configurable Dropout ---
    # We extract dropout from config to test regularization strength
    d_rate = config.get('dropout', 0.5)
    model = model_builder(dropout_rate=d_rate)

    # --- C. Compile with Configurable Learning Rate ---
    # We allows tuning LR for deeper models (like ResNet) that need stability
    lr = config.get('learning_rate', 0.001)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    # --- D. Train with Augmentation ---
    # Augmentation prevents overfitting by creating variations of images
    datagen = ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    print(f"\nStarting Experiment: {exp_name}")
    history = model.fit(
        datagen.flow(x_train, y_train, batch_size=config['batch_size'], seed=SEED),
        epochs=config['epochs'],
        validation_data=(x_val, y_val),
        class_weight=class_weights, # Apply imbalance fix
        callbacks=[WandbMetricsLogger()], # Auto-log loss/accuracy to W&B
        verbose=1
    )

    # --- E. Evaluate on Test Set ---
    print("\nGenerating Test Predictions...")
    y_pred_probs = model.predict(x_test)
    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)

    # --- F. Log Advanced Metrics ---
    # We calculate Recall/Precision specifically for the "Illness" class (1)
    test_acc = accuracy_score(y_true_classes, y_pred_classes)
    test_auc = roc_auc_score(y_test, y_pred_probs)

    report_dict = classification_report(
        y_true_classes,
        y_pred_classes,
        target_names=['Benign', 'Illness'],
        output_dict=True
    )

    # Log scalar values to W&B Table
    wandb.log({
        "test_accuracy": test_acc,
        "test_auc": test_auc,
        "test_recall_illness": report_dict['Illness']['recall'],
        "test_precision_illness": report_dict['Illness']['precision']
    })

    # --- G. Log Interactive Visualizations ---
    # ROC Curve: Shows trade-off between sensitivity and false positives
    wandb.log({"roc_curve": wandb.plot.roc_curve(
        y_true_classes,
        y_pred_probs,
        labels=["Benign", "Illness"]
    )})

    # Confusion Matrix: Shows exactly where mistakes happen
    wandb.log({"conf_mat": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_true_classes,
        preds=y_pred_classes,
        class_names=["Benign", "Illness"]
    )})

    # Print final report to console for quick check
    print("\nFinal Test Report:")
    print(classification_report(y_true_classes, y_pred_classes, target_names=['Benign', 'Illness']))

    wandb.finish()
    print(f"Experiment {exp_name} finished successfully.")

Loading DermaMNIST...


In [8]:
# ==========================================
# 5. MODEL ARCHITECTURES
# ==========================================

# --- MODEL 1: THE BASELINE CNN ---
# A simple, standard architecture to establish a performance benchmark.
# It uses single convolution layers followed by pooling.
def build_baseline_cnn(dropout_rate=0.5):
    # We use a lower dropout rate for convolutional layers to avoid losing too much spatial info.
    conv_drop = dropout_rate * 0.5

    model = models.Sequential([
        layers.Input(shape=(28, 28, 3)),

        # Layer 1: Detects basic edges/colors
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(conv_drop),

        # Layer 2: Detects simple shapes
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(conv_drop),

        # Layer 3: Detects complex textures
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        # Classifier Head
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(dropout_rate), # Stronger dropout before final classification
        layers.Dense(2, activation='softmax')
    ])
    return model

# --- MODEL 2: VGG-STYLE (DEEP) ---
# Mimics the famous VGG network by stacking two convolutions before every pooling layer.
# This increases the "receptive field" and non-linearity without shrinking the image too fast.
def build_vgg_style(dropout_rate=0.5):
    conv_drop = dropout_rate * 0.5

    model = models.Sequential([
        layers.Input(shape=(28, 28, 3)),

        # Block 1: Double Conv for better feature extraction at high resolution
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(conv_drop),

        # Block 2: Double Conv for mid-level features
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(conv_drop),

        # Classifier Head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(2, activation='softmax')
    ])
    return model

# --- MODEL 3: RESNET-STYLE (MODERN) ---
# Uses "Skip Connections" (shortcuts) to allow gradients to flow easily.
# This prevents the "vanishing gradient" problem and is very stable.
def build_resnet_style(dropout_rate=0.5):

    # Helper function for a Residual Block
    # Output = Activation( Conv(x) + Shortcut(x) )
    def res_block(x, filters, stride=1):
        shortcut = x

        # Main Path
        x = layers.Conv2D(filters, (3, 3), strides=stride, padding='same')(x)
        x = layers.BatchNormalization()(x) # BN is crucial for ResNet stability
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)

        # Shortcut Path: Resize input if dimensions change
        if stride > 1 or shortcut.shape[-1] != filters:
            shortcut = layers.Conv2D(filters, (1, 1), strides=stride, padding='same')(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)

        # The Skip Connection: Add original input to the processed output
        x = layers.Add()([x, shortcut])
        x = layers.Activation('relu')(x)
        return x

    inputs = layers.

In [ ]:
# ==========================================
# 6. RUNNING THE EXPERIMENTS
# ==========================================

# --- Experiment 1: Baseline (Reference) ---
# Already provided, but included here for completeness
train_experiment(
    model_builder=build_baseline_cnn,
    exp_name="Exp1_Baseline",
    config={
        "batch_size": 128,
        "epochs": 30,
        "learning_rate": 0.001, # Standard speed
        "dropout": 0.25         # Light regularization
    }
)

In [ ]:
# --- Experiment 2: VGG-Style (The "Deep" Candidate) ---
# VGG is very deep and "hungry" for data. We lower the LR to prevent
# spiky loss and increase dropout to fight overfitting.
train_experiment(
    model_builder=build_vgg_style,
    exp_name="Exp2_VGG_Deep",
    config={
        "batch_size": 128,
        "epochs": 40,           # Needs more time to converge
        "learning_rate": 0.0001, # 10x slower for stability
        "dropout": 0.5          # Stronger regularization
    }
)

In [ ]:
# --- Experiment 3: ResNet-Style (The "Modern" Candidate) ---
# ResNet is naturally stable due to skip connections.
# We can often afford a slightly higher LR than VGG.
train_experiment(
    model_builder=build_resnet_style,
    exp_name="Exp3_ResNet_Custom",
    config={
        "batch_size": 128,
        "epochs": 30,
        "learning_rate": 0.0005, # Balanced speed
        "dropout": 0.5           # Standard regularization
    }
)